In [2]:
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification


d:\anaconda\envs\finbert-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from huggingface_hub import login
login("")

In [4]:
dataset = load_dataset('kurry/sp500_earnings_transcripts')
print(dataset['train'][0])

{'symbol': 'A', 'quarter': 4, 'year': 2020, 'date': '2020-11-23 16:30:00', 'content': "Operator: Good afternoon, and welcome to the Agilent Technologies Fourth Quarter Earnings Conference Call. All lines have been placed on mute to prevent any background noise. After the speakers' remarks, there will be a question-and-answer session. [Operator Instructions] Thank you. And now, I'd like to introduce you to the host for today's conference, Ankur Dhingra, Vice President of Investor Relations. Sir, please go ahead.\nAnkur Dhingra: Thank you, and welcome everyone to Agilent's fourth quarter and full-year conference call for fiscal year 2020. With me are Mike McMullen, Agilent's President and CEO; and Bob McMahon, Agilent's Senior Vice President and CFO. Joining in the Q&A after Bob's comments will be: Jacob Thaysen, President of Agilent's Life Sciences & Applied Markets Group; Sam Raha, President of Agilent's Diagnostics and Genomics Group; and Padraig McDonnell, President of Agilent CrossL

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id'],
        num_rows: 33362
    })
})

In [5]:
import re
def remove_disclaimer(text):
    disclaimer_patterns = {
        r"forward[- ]looking statements.*?(?=\n\n)",
        r"safe harbor.*?(?=\n\n)",
        r"certain statements.*?risks and uncertainties.*?(?=\n\n)"
    }
    for pattern in disclaimer_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.DOTALL) # to remove multi-line disclaimer blocks
    return text

def clean_text(text):
    text = re.sub(r"\[.*?\]", "", text)   # remove bracket content
    text = re.sub(r"\s+", " ", text)      # normalize whitespace
    return text.strip()

In [ ]:
def extract_management_text(record):
    structured = record["structured_content"]
    
    segments = []
    
    for seg in structured:
        speaker = seg.get("speaker", "")
        text = seg.get("text", "")
        
        # remove operator speech
        if speaker and speaker.lower() == "operator":
            continue
        
        # tp rremove extremely short segments
        if len(text) < 50:
            continue
        
        segments.append(text)
    
    combined_text = " ".join(segments)
    combined_text = remove_disclaimer(combined_text)
    combined_text = clean_text(combined_text)
    
    return {"processed_text": combined_text}

In [ ]:
dataset1 = dataset.map(extract_management_text)
dataset1 = dataset1.filter(lambda x: len(x['processed_text']) > 1000)
print(dataset1['train'][0]['processed_text'][:500])  

Thank you, and welcome everyone to Agilent's fourth quarter and full-year conference call for fiscal year 2020. With me are Mike McMullen, Agilent's President and CEO; and Bob McMahon, Agilent's Senior Vice President and CFO. Joining in the Q&A after Bob's comments will be: Jacob Thaysen, President of Agilent's Life Sciences & Applied Markets Group; Sam Raha, President of Agilent's Diagnostics and Genomics Group; and Padraig McDonnell, President of Agilent CrossLab Group. This presentation is be


In [8]:
dataset1['train'][0]

{'symbol': 'A',
 'quarter': 4,
 'year': 2020,
 'date': '2020-11-23 16:30:00',
 'content': "Operator: Good afternoon, and welcome to the Agilent Technologies Fourth Quarter Earnings Conference Call. All lines have been placed on mute to prevent any background noise. After the speakers' remarks, there will be a question-and-answer session. [Operator Instructions] Thank you. And now, I'd like to introduce you to the host for today's conference, Ankur Dhingra, Vice President of Investor Relations. Sir, please go ahead.\nAnkur Dhingra: Thank you, and welcome everyone to Agilent's fourth quarter and full-year conference call for fiscal year 2020. With me are Mike McMullen, Agilent's President and CEO; and Bob McMahon, Agilent's Senior Vice President and CFO. Joining in the Q&A after Bob's comments will be: Jacob Thaysen, President of Agilent's Life Sciences & Applied Markets Group; Sam Raha, President of Agilent's Diagnostics and Genomics Group; and Padraig McDonnell, President of Agilent Cr

In [9]:
dataset1 = dataset1.remove_columns([
    "symbol",
    "company_name",
    "company_id",
    "year",
    "quarter",
    "date",
    "content",
    "structured_content"
])

In [10]:
dataset1

DatasetDict({
    train: Dataset({
        features: ['processed_text'],
        num_rows: 33216
    })
})

## labeling & tokenizer

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
dataset1.save_to_disk("dataset1_earnings_dataset")

Saving the dataset (0/4 shards):   0%|          | 0/33216 [00:00<?, ? examples/s]

Saving the dataset (4/4 shards): 100%|██████████| 33216/33216 [00:02<00:00, 14457.41 examples/s]


In [14]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
sentiment_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
uncertainty_words = [
    "may", "might", "could", "uncertain", "risk",
    "challenging", "volatility", "pressure",
    "decline", "weakness", "headwind",
    "concern", "slowdown", "impact"
]

def uncertainty_score(text):
    text = text.lower().split()
    total_words = len(text)

    if total_words == 0:
        return 0.0

    count = sum(word in uncertainty_words for word in text)

    return count / total_words

In [ ]:
sentiment_model.to(device)
def add_simple_label(example):
    heuristic = uncertainty_score(example["processed_text"])
    heuristic = min(heuristic * 20, 1.0)
    return {"risk_score": heuristic}

dataset_labeled = dataset1.map(add_simple_label)



Map: 100%|██████████| 33216/33216 [01:03<00:00, 522.58 examples/s]


In [ ]:
def tokenize_and_chunk(examples):
    tokenized = tokenizer(
        examples["processed_text"],
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding="max_length",
        return_offsets_mapping=False
    )

    overflow_mapping = tokenized.pop("overflow_to_sample_mapping")

    risk_scores = []

    for i, input_ids in enumerate(tokenized["input_ids"]):
        # decode this chunk back to text
        chunk_text = tokenizer.decode(input_ids, skip_special_tokens=True)

        # compute risk on THIS chunk only
        heuristic = uncertainty_score(chunk_text)
        heuristic = min(heuristic * 20, 1.0)

        risk_scores.append(heuristic)

    tokenized["risk_score"] = risk_scores

    return tokenized

In [18]:
dataset_final = dataset1.map(
    tokenize_and_chunk,
    batched=True,
    remove_columns=["processed_text"]
)

Map: 100%|██████████| 33216/33216 [1:03:07<00:00,  8.77 examples/s]


In [23]:
dataset_final.save_to_disk("tokenized_earnings_dataset")

Saving the dataset (5/5 shards): 100%|██████████| 921875/921875 [00:05<00:00, 159509.74 examples/s]


In [19]:
dataset_final

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'risk_score'],
        num_rows: 921875
    })
})

## Training

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#  Subsample & Train-validation split
full_dataset = dataset_final['train'].shuffle(seed=42).select(range(50000))  # only 30k samples


# 90% train, 10% val
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_set = full_dataset.select(range(train_size))
val_set = full_dataset.select(range(train_size, train_size + val_size))

train_set.set_format('torch')
val_set.set_format('torch')


# Dataloaders
batch_size = 16
train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


#  Model
class BertReg(nn.Module):
    def __init__(self):
        super().__init__()
        self.modell = AutoModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.regressor = nn.Linear(self.modell.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask): #token_type_ids=None):
        outputs = self.modell(
            input_ids=input_ids,
            attention_mask=attention_mask,
            # token_type_ids=token_type_ids
        )
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_embedding)
        x = self.regressor(x)
        return x.squeeze(-1)

model = BertReg().to(device)


# Optimizer
optimizer = torch.optim.AdamW([
    {'params': model.modell.parameters(), 'lr': 1e-5},
    {'params': model.regressor.parameters(), 'lr': 3e-5}
], weight_decay=0.01)


# Scheduler
epochs = 3
total_steps = len(train_loader) * epochs
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
criterion = nn.SmoothL1Loss()
scaler = torch.amp.GradScaler()


#Training Function
def train_one_epoch(model, train_loader, epoch):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["risk_score"].float().to(device)
        # token_type_ids = batch["token_type_ids"].to(device)

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids, attention_mask,) #token_type_ids)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] "
                  f"Step [{step+1}/{len(train_loader)}] "
                  f"Train Loss: {loss.item():.4f}")
            
    return total_loss / len(train_loader)


#Evaluation
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    preds, true_vals = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["risk_score"].float().to(device)
            # token_type_ids = batch["token_type_ids"].to(device)

            outputs = model(input_ids, attention_mask,) #token_type_ids)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds.extend(outputs.cpu().numpy())
            true_vals.extend(labels.cpu().numpy())

    mse = mean_squared_error(true_vals, preds)
    mae = mean_absolute_error(true_vals, preds)
    r2 = r2_score(true_vals, preds)

    return total_loss / len(loader), mse, mae, r2


Using device: cuda


In [21]:
for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, epoch)
    val_loss, mse, mae, r2 = evaluate(model, val_loader)

    print(f"\n==== Epoch {epoch+1} Complete ====")
    print(f"Avg Train Loss: {train_loss:.4f}")
    print(f"Final Val Loss: {val_loss:.4f}")
    print(f"MSE: {mse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

Epoch [1/6] Step [50/2813] Train Loss: 0.0417
Epoch [1/6] Step [100/2813] Train Loss: 0.0241
Epoch [1/6] Step [150/2813] Train Loss: 0.0144
Epoch [1/6] Step [200/2813] Train Loss: 0.0154
Epoch [1/6] Step [250/2813] Train Loss: 0.0093
Epoch [1/6] Step [300/2813] Train Loss: 0.0097
Epoch [1/6] Step [350/2813] Train Loss: 0.0067
Epoch [1/6] Step [400/2813] Train Loss: 0.0078
Epoch [1/6] Step [450/2813] Train Loss: 0.0137
Epoch [1/6] Step [500/2813] Train Loss: 0.0031
Epoch [1/6] Step [550/2813] Train Loss: 0.0031
Epoch [1/6] Step [600/2813] Train Loss: 0.0069
Epoch [1/6] Step [650/2813] Train Loss: 0.0036
Epoch [1/6] Step [700/2813] Train Loss: 0.0080
Epoch [1/6] Step [750/2813] Train Loss: 0.0013
Epoch [1/6] Step [800/2813] Train Loss: 0.0038
Epoch [1/6] Step [850/2813] Train Loss: 0.0043
Epoch [1/6] Step [900/2813] Train Loss: 0.0043
Epoch [1/6] Step [950/2813] Train Loss: 0.0020
Epoch [1/6] Step [1000/2813] Train Loss: 0.0054
Epoch [1/6] Step [1050/2813] Train Loss: 0.0069
Epoch [1/6] 

KeyboardInterrupt: 

In [22]:
torch.save(model.state_dict(), "distilbert_risk_model.pth")